# Fine-Tuning Phi on AML Compute Instance (GPU)

This notebook fine-tunes **Phi-4-mini-instruct** on an Azure ML Compute Instance with a T4 GPU using QLoRA + SFTTrainer.

Key techniques (adapted from `fine_tune_llama_3_doctor.py`):
- **QLoRA** 4-bit quantization (NF4, double quant) via `bitsandbytes`
- **LoRA** with `r=8`, `alpha=32` via `peft`
- **Gradient checkpointing** to reduce VRAM usage
- **TF32 / BF16** auto-detection for Ampere+ GPUs
- **Memory optimization**: expandable CUDA segments, paged AdamW, pinned memory disabled
- **MLflow** logging to Azure ML
- **SFTTrainer** from `trl`

### Prerequisites
1. Compute Instance created with custom `sft-notebook` conda env (see `aml_ci_create.ipynb`)
2. `.env` file with `SUBSCRIPTION_ID`, `RESOURCE_GROUP`, `WORKSPACE`, `SSH_PUB_KEY_NAME`
3. Training data in `data/train.jsonl` and `data/validation.jsonl` (chat-completion JSONL format)

## 1. Environment Setup & GPU Check

In [1]:
from applyllm.accelerators import (
    AcceleratorHelper,
    AcceleratorStatus,
    DirectorySetting,
)

# Configure applyllm DirectorySetting to store HF models in localfiles/models
aml_dir_setting = DirectorySetting(
    home_dir="/home/azureuser",
    transformers_cache_home="localfiles/models",
    huggingface_token_file="localfiles/models/.huggingface_token",
)
uuids = AcceleratorHelper.nvidia_device_uuids_filtered_by(is_mig=False)
AcceleratorHelper.init_torch_env(
    accelerator="cuda", dir_setting=aml_dir_setting, uuids=uuids
)

/anaconda/envs/sft-notebook/lib/python3.14/site-packages/langchain_core/_api/deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


In [2]:
import os
import gc
import torch
import mlflow
from accelerate import Accelerator

from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
)
from peft import (
    LoraConfig,
    prepare_model_for_kbit_training,
    get_peft_model,
)
from datasets import load_dataset
from trl import SFTTrainer

# Additional memory optimization (overrides applyllm default max_split_size_mb)
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# TF32 on Ampere+ for speed
if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"XDG_CACHE_HOME: {os.environ.get('XDG_CACHE_HOME', 'not set')}")
print(f"Model cache dir: {aml_dir_setting.get_cache_home()}")

# Display GPU info via applyllm AcceleratorStatus
acc_status = AcceleratorStatus.create_accelerator_status()
acc_status.gpu_usage()

PyTorch version: 2.11.0+cu126
CUDA available: True
XDG_CACHE_HOME: /home/azureuser/localfiles/models
Model cache dir: /home/azureuser/localfiles/models
num_of_gpus: 1
--------------------
Device name      : Tesla T4 
Device idx       : 0 
No. of processors: 40
Physical  memory : 15.574707 GB
Reserved  memory : 0.000000 GB
Allocated memory : 0.000000 GB
Free      memory : 0.000000 GB
--------------------


## 2. Authentication & Configuration

In [3]:
import sys
from pathlib import Path
from dotenv import load_dotenv
from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential

# Ensure the notebook directory is on sys.path so "utils" is importable
nb_dir = str(Path("/home/azureuser/localfiles/model-fine-tuning/01-compute-instance").resolve())
if nb_dir not in sys.path:
    sys.path.insert(0, nb_dir)

load_dotenv(dotenv_path=".env", override=True)

credential = DefaultAzureCredential()

# On a Compute Instance the workspace config.json is auto-provisioned
try:
    ml_client = MLClient.from_config(credential=credential)
    print(f"Connected to workspace: {ml_client.workspace_name}")
except Exception:
    # Fallback: use .env settings via amlauth helper
    from utils.amlauth import AuthHelper
    settings = AuthHelper.load_settings()
    ml_client = MLClient(
        credential=credential,
        subscription_id=settings.subscription_id,
        resource_group_name=settings.resource_group,
        workspace_name=settings.workspace,
    )
    print(f"Connected to workspace: {ml_client.workspace_name} (from .env)")

# Configure MLflow to track to AML workspace
tracking_uri = ml_client.workspaces.get(ml_client.workspace_name).mlflow_tracking_uri
mlflow.set_tracking_uri(tracking_uri)
print(f"MLflow tracking URI: {tracking_uri}")

Found the config file in: /config.json
Class DeploymentTemplateOperations: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.


Connected to workspace: aml-workspace-yw-uno
MLflow tracking URI: azureml://swedencentral.api.azureml.ms/mlflow/v2.0/subscriptions/6753a2ee-12b7-4fac-82fa-48824fb58abe/resourceGroups/rg-aml-yw-uno/providers/Microsoft.MachineLearningServices/workspaces/aml-workspace-yw-uno


## 3. Training Parameters

In [16]:
# Model & dataset configuration
BASE_MODEL = "microsoft/Phi-4-mini-instruct"
FINETUNED_MODEL = "phi-4-mini-instruct-finetuned"

# HF cache directory from applyllm DirectorySetting → /home/azureuser/localfiles/models
HF_CACHE = aml_dir_setting.get_cache_home()
os.makedirs(HF_CACHE, exist_ok=True)

# Dataset from HuggingFace Hub
FINETUNE_DATASET = "ruslanmv/ai-medical-chatbot"
NUM_DATA_ROWS = 1000
EVAL_SIZE = 0.1

# Training hyperparameters
NUM_EPOCHS = 1
MAX_STEPS = -1          # set to a small number (e.g. 20) for quick testing, -1 = use NUM_EPOCHS
BATCH_SIZE = 1
LEARNING_RATE = 2e-4
MAX_SEQ_LENGTH = 512
GRADIENT_ACCUMULATION_STEPS = 8
WARMUP_STEPS = 10
LOGGING_STEPS = 50
EVAL_STEPS = 100
SAVE_STEPS = 500

# LoRA hyperparameters
LORA_R = 8
LORA_ALPHA = 32
LORA_DROPOUT = 0.05

# Output directories
FT_MODELS_DIR = "/home/azureuser/localfiles/finetunedmodels"
os.makedirs(FT_MODELS_DIR, exist_ok=True)


print(f"HF model cache: {HF_CACHE}")
print("SFT Configuration loaded")

print(f"Dataset: {FINETUNE_DATASET}")
print(f"Epochs: {NUM_EPOCHS}, Max steps: {MAX_STEPS}")
print(f"Fine-tuned models dir: {FT_MODELS_DIR}")

HF model cache: /home/azureuser/localfiles/models
SFT Configuration loaded
Dataset: ruslanmv/ai-medical-chatbot
Epochs: 1, Max steps: -1
Fine-tuned models dir: /home/azureuser/localfiles/finetunedmodels


## 4. Prepare Dataset

Load `ruslanmv/ai-medical-chatbot` from HuggingFace Hub, shuffle, select `NUM_DATA_ROWS` samples, and split into train/eval sets. The dataset has `Patient` and `Doctor` columns which are formatted into a chat template.

In [5]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, cache_dir=HF_CACHE)
tokenizer.model_max_length = MAX_SEQ_LENGTH

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Load dataset from HuggingFace Hub
print(f"Loading HuggingFace dataset: {FINETUNE_DATASET}")
full_ds = load_dataset(FINETUNE_DATASET, split="all", cache_dir=HF_CACHE)
full_ds = full_ds.shuffle(seed=65).select(range(min(NUM_DATA_ROWS, len(full_ds))))
dataset_split = full_ds.train_test_split(test_size=EVAL_SIZE, seed=65)
dataset = {"train": dataset_split["train"], "test": dataset_split["test"]}

def format_chat_template(row):
    """Format Patient/Doctor columns into chat template."""
    messages = [
        {"role": "user", "content": row["Patient"]},
        {"role": "assistant", "content": row["Doctor"]},
    ]
    row["text"] = tokenizer.apply_chat_template(messages, tokenize=False)
    return row

dataset["train"] = dataset["train"].map(format_chat_template, num_proc=4)
dataset["test"] = dataset["test"].map(format_chat_template, num_proc=4)

print(f"Train samples: {len(dataset['train'])}, Eval samples: {len(dataset['test'])}")
print(f"\nSample:\n{dataset['train'][0]['text'][:500]}")

This model config has set a `rope_parameters['original_max_position_embeddings']` field, to be used together with `max_position_embeddings` to determine a scaling factor. Please set the `factor` field of `rope_parameters`with this ratio instead -- we recommend the use of this field over `original_max_position_embeddings`, as it is compatible with most model architectures.


Loading HuggingFace dataset: ruslanmv/ai-medical-chatbot


Train samples: 900, Eval samples: 100

Sample:
<|user|>Hi I was so confused please help....for a time bow I noticed that whenever I eat the Tomally (crab innards) I get very itchy all over my skin and I develop spots. So I thought I developed a seafood allergy..however I noticed that I can crab meat no problem but once I eat the innards I get itchy...what is causing this? What is in the innards to make me allergic? I have eaten scallops lobster etc an no issues. Please advise Thanks you<|end|><|assistant|>HI, thanks for using healthcare magi


## 5. Load Model with QLoRA Quantization

In [6]:
# Auto-detect bf16 support (Ampere+ / compute capability >= 8.0)
use_bf16 = False
if torch.cuda.is_available():
    major_cc, _ = torch.cuda.get_device_capability(0)
    use_bf16 = major_cc >= 8
    print(f"BF16 support: {use_bf16} (compute capability {major_cc})")

# QLoRA 4-bit configuration
qlora_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16 if use_bf16 else torch.float16,
    bnb_4bit_use_double_quant=True,
)

# Device mapping
if torch.cuda.is_available():
    device_index = Accelerator().process_index
    device_map = {"": device_index}
else:
    device_map = "auto"
    qlora_config = None

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=qlora_config,
    device_map=device_map,
    attn_implementation="eager",
    cache_dir=HF_CACHE,
)

# Prepare model for quantized training
model = prepare_model_for_kbit_training(model)

# Disable KV cache (incompatible with gradient checkpointing)
if hasattr(model, "config"):
    model.config.use_cache = False

# Ensure inputs require grads for checkpointing
if hasattr(model, "enable_input_require_grads"):
    model.enable_input_require_grads()

print(f"Model loaded: {BASE_MODEL}")
print(f"Model parameters: {model.num_parameters():,}")

BF16 support: False (compute capability 7)


Loading weights:   0%|          | 0/194 [00:00<?, ?it/s]

/anaconda/envs/sft-notebook/lib/python3.14/site-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Model loaded: microsoft/Phi-4-mini-instruct
Model parameters: 3,836,021,760


## 6. Configure LoRA & Apply to Model

In [7]:
peft_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["qkv_proj", "o_proj", "gate_up_proj", "down_proj"],
)

model = get_peft_model(model, peft_config)

# Enable gradient checkpointing to save VRAM
model.gradient_checkpointing_enable()

# Clear CUDA cache before training
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.synchronize()
    gc.collect()

model.print_trainable_parameters()

trainable params: 11,534,336 || all params: 3,847,556,096 || trainable%: 0.2998


## 7. Training

In [ ]:
# AML MLflow backend rejects param values > 500 chars.
# Patch mlflow.log_params to truncate before sending to AML.
# Guard ensures re-running the cell doesn't create infinite recursion.
_AML_MAX_PARAM_LEN = 500
if not getattr(mlflow.log_params, "_is_truncated_patch", False):
    _original_mlflow_log_params = mlflow.log_params

    def _truncated_log_params(params, **kwargs):
        truncated = {}
        for k, v in params.items():
            s = str(v)
            truncated[k] = s[:_AML_MAX_PARAM_LEN] if len(s) > _AML_MAX_PARAM_LEN else v
        return _original_mlflow_log_params(truncated, **kwargs)

    _truncated_log_params._is_truncated_patch = True
    mlflow.log_params = _truncated_log_params

training_args = TrainingArguments(
    output_dir=os.path.join(FT_MODELS_DIR, FINETUNED_MODEL),
    per_device_train_batch_size=max(1, BATCH_SIZE),
    per_device_eval_batch_size=max(1, BATCH_SIZE),
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    optim="paged_adamw_32bit" if torch.cuda.is_available() else "adamw_torch",
    num_train_epochs=NUM_EPOCHS,
    max_steps=MAX_STEPS,
    eval_strategy="steps",
    eval_steps=EVAL_STEPS,
    logging_steps=LOGGING_STEPS,
    warmup_steps=WARMUP_STEPS,
    logging_strategy="steps",
    learning_rate=LEARNING_RATE,
    fp16=False,
    bf16=use_bf16,
    gradient_checkpointing=True,
    report_to="mlflow",
    dataloader_pin_memory=False,
    save_steps=SAVE_STEPS,
    eval_accumulation_steps=1,
    ddp_find_unused_parameters=False,
    dataloader_num_workers=2,
    disable_tqdm=True,
)

# Remove NotebookProgressCallback which crashes in VS Code Remote SSH
from transformers.utils.notebook import NotebookProgressCallback

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    processing_class=tokenizer,
    args=training_args,
)
trainer.remove_callback(NotebookProgressCallback)

print("Starting training...")
trainer.train()

Starting training...


{'loss': '3.117', 'grad_norm': '0.1377', 'learning_rate': '0.0001243', 'entropy': '2.941', 'num_tokens': '9.124e+04', 'mean_token_accuracy': '0.3884', 'epoch': '0.4444'}
{'loss': '2.841', 'grad_norm': '0.1118', 'learning_rate': '2.718e-05', 'entropy': '2.856', 'num_tokens': '1.753e+05', 'mean_token_accuracy': '0.4116', 'epoch': '0.8889'}
{'eval_loss': '2.743', 'eval_runtime': '49.5', 'eval_samples_per_second': '2.02', 'eval_steps_per_second': '2.02', 'eval_entropy': '2.814', 'eval_num_tokens': '1.753e+05', 'eval_mean_token_accuracy': '0.4243', 'epoch': '0.8889'}
{'eval_loss': '2.741', 'eval_runtime': '49.18', 'eval_samples_per_second': '2.034', 'eval_steps_per_second': '2.034', 'eval_entropy': '2.818', 'eval_num_tokens': '1.985e+05', 'eval_mean_token_accuracy': '0.4247', 'epoch': '1'}


config.json: 0.00B [00:00, ?B/s]

{'train_runtime': '1145', 'train_samples_per_second': '0.786', 'train_steps_per_second': '0.099', 'train_loss': '2.964', 'epoch': '1'}
🏃 View run calm_ball_7dp0ct3m at: https://swedencentral.api.azureml.ms/mlflow/v2.0/subscriptions/6753a2ee-12b7-4fac-82fa-48824fb58abe/resourceGroups/rg-aml-yw-uno/providers/Microsoft.MachineLearningServices/workspaces/aml-workspace-yw-uno/#/experiments/1d65e0dc-0eb5-49d5-bab6-6dbb2a91e6d6/runs/03a81355-f423-40bc-b665-b9b8792018bd
🧪 View experiment at: https://swedencentral.api.azureml.ms/mlflow/v2.0/subscriptions/6753a2ee-12b7-4fac-82fa-48824fb58abe/resourceGroups/rg-aml-yw-uno/providers/Microsoft.MachineLearningServices/workspaces/aml-workspace-yw-uno/#/experiments/1d65e0dc-0eb5-49d5-bab6-6dbb2a91e6d6


TrainOutput(global_step=113, training_loss=2.9639623861397264, metrics={'train_runtime': 1144.9849, 'train_samples_per_second': 0.786, 'train_steps_per_second': 0.099, 'total_flos': 3850648727777280.0, 'train_loss': 2.9639623861397264})

In [10]:
# Quick check: training loss trend
log_history = trainer.state.log_history
train_logs = [l for l in log_history if "loss" in l]
eval_logs = [l for l in log_history if "eval_loss" in l]

print(f"Total training log entries: {len(train_logs)}")
print(f"Total eval log entries: {len(eval_logs)}")
print("\n--- Training Loss ---")
for l in train_logs[:5]:
    print(f"  Step {l['step']:>4}: loss = {l['loss']:.4f}")
if len(train_logs) > 5:
    print(f"  ...")
    for l in train_logs[-3:]:
        print(f"  Step {l['step']:>4}: loss = {l['loss']:.4f}")

if eval_logs:
    print("\n--- Eval Loss ---")
    for l in eval_logs:
        print(f"  Step {l['step']:>4}: eval_loss = {l['eval_loss']:.4f}")

if train_logs:
    first_loss = train_logs[0]["loss"]
    last_loss = train_logs[-1]["loss"]
    print(f"\nFirst train loss: {first_loss:.4f} → Last train loss: {last_loss:.4f} (Δ = {last_loss - first_loss:+.4f})")

Total training log entries: 2
Total eval log entries: 2

--- Training Loss ---
  Step   50: loss = 3.1170
  Step  100: loss = 2.8415

--- Eval Loss ---
  Step  100: eval_loss = 2.7426
  Step  113: eval_loss = 2.7409

First train loss: 3.1170 → Last train loss: 2.8415 (Δ = -0.2755)


## 8. Save Model

Save the LoRA adapter, the LoRA config, and the merged full model.

In [17]:
# Save LoRA adapter + tokenizer (same dir as training output_dir)
adapter_path = os.path.join(FT_MODELS_DIR, FINETUNED_MODEL)
trainer.save_model(adapter_path)
tokenizer.save_pretrained(adapter_path)
print(f"LoRA adapter + tokenizer saved to: {adapter_path}")

# Save LoRA config
config_path = os.path.join(FT_MODELS_DIR, f"{FINETUNED_MODEL}_config")
peft_config.save_pretrained(config_path)
print(f"LoRA config saved to: {config_path}")

# Merge LoRA weights into base model and save full model + tokenizer
full_model_path = os.path.join(FT_MODELS_DIR, f"{FINETUNED_MODEL}_full")
model.merge_and_unload().save_pretrained(full_model_path)
tokenizer.save_pretrained(full_model_path)
print(f"Merged full model + tokenizer saved to: {full_model_path}")

LoRA adapter + tokenizer saved to: /home/azureuser/localfiles/finetunedmodels/phi-4-mini-instruct-finetuned
LoRA config saved to: /home/azureuser/localfiles/finetunedmodels/phi-4-mini-instruct-finetuned_config


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Merged full model + tokenizer saved to: /home/azureuser/localfiles/finetunedmodels/phi-4-mini-instruct-finetuned_full


## 9. Test Inference

In [18]:
# Load the saved tokenizer from the adapter output
adapter_path = os.path.join(FT_MODELS_DIR, FINETUNED_MODEL)
ft_tokenizer = AutoTokenizer.from_pretrained(adapter_path)

messages = [
    {"role": "user", "content": "Hello, I have been experiencing headaches and dizziness for the past week. What could be causing this?"}
]

prompt = ft_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
device = next(model.parameters()).device
inputs = ft_tokenizer(prompt, return_tensors="pt", padding=True, truncation=True).to(device)

outputs = model.generate(**inputs, max_new_tokens=200, num_return_sequences=1)
response = ft_tokenizer.decode(outputs[0], skip_special_tokens=False)

# Format output to separate user prompt from assistant response
parts = response.split("<|assistant|>")
if len(parts) >= 2:
    user_part = parts[0].replace("<|user|>", "").replace("<|end|>", "").strip()
    assistant_part = parts[-1].replace("<|end|>", "").strip()
    print("=" * 60)
    print("USER:")
    print(f"  {user_part}")
    print("-" * 60)
    print("ASSISTANT:")
    print(f"  {assistant_part}")
    print("=" * 60)
else:
    print(response)

/anaconda/envs/sft-notebook/lib/python3.14/site-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


USER:
  Hello, I have been experiencing headaches and dizziness for the past week. What could be causing this?
------------------------------------------------------------
ASSISTANT:
  I'm sorry to hear that you're experiencing these symptoms. Headaches and dizziness can be caused by a variety of factors, and it's important to consider the whole picture. Some common causes include dehydration, tension headaches, migraines, ear infections, vision problems, or even more serious conditions like brain tumors or vascular issues. However, I am not a doctor, and my assessment cannot replace professional medical advice. It's crucial to consult with a healthcare provider to determine the cause of your symptoms and receive appropriate treatment. In the meantime, ensure you stay hydrated, get plenty of rest, and avoid any known headache triggers. If your symptoms persist or worsen, please seek medical attention promptly.


## 10. Register Model in AML

Register the fine-tuned model in the Azure ML workspace for deployment.

In [14]:
from azure.ai.ml.entities import Model
from azure.ai.ml.constants import AssetTypes

full_model_path = os.path.join(FT_MODELS_DIR, f"{FINETUNED_MODEL}_full")

# Build description with training details
description = (
    f"Fine-tuned {BASE_MODEL} with QLoRA + SFTTrainer\n"
    f"Dataset: {FINETUNE_DATASET} ({NUM_DATA_ROWS} samples)\n"
    f"LoRA: r={LORA_R}, alpha={LORA_ALPHA}, dropout={LORA_DROPOUT}\n"
    f"Training: epochs={NUM_EPOCHS}, lr={LEARNING_RATE}, batch={BATCH_SIZE}x{GRADIENT_ACCUMULATION_STEPS} grad_accum"
)

# Add final loss info if available
log_history = trainer.state.log_history
train_logs = [l for l in log_history if "loss" in l]
eval_logs = [l for l in log_history if "eval_loss" in l]
if train_logs:
    description += f"\nFinal train loss: {train_logs[-1]['loss']:.4f}"
if eval_logs:
    description += f"\nFinal eval loss: {eval_logs[-1]['eval_loss']:.4f}"

tags = {
    "base_model": BASE_MODEL,
    "dataset": FINETUNE_DATASET,
    "lora_r": str(LORA_R),
    "lora_alpha": str(LORA_ALPHA),
    "num_epochs": str(NUM_EPOCHS),
    "quantization": "QLoRA-4bit-NF4",
}

registered_model = ml_client.models.create_or_update(
    Model(
        path=full_model_path,
        name=FINETUNED_MODEL,
        type=AssetTypes.CUSTOM_MODEL,
        description=description,
        tags=tags,
    )
)

print(f"Registered model: {registered_model.name}")
print(f"Version: {registered_model.version}")
print(f"ID: {registered_model.id}")
print(f"\nDescription:\n{description}")
print(f"\nTags: {tags}")

Your file exceeds 100 MB. If you experience low speeds, latency, or broken connections, we recommend using the AzCopyv10 tool for this file transfer.

Example: azcopy copy '/home/azureuser/localfiles/finetunedmodels/phi-4-mini-instruct-finetuned_full' 'https://amlworkspaceyw3405831058.blob.core.windows.net/azureml-blobstore-bae0a300-6d85-4254-8b17-790d9c6d7a5e/LocalUpload/a6e242423307bbd30a125d15717b4c82a03f954af6d496a170ecb05d24a09b4e/phi-4-mini-instruct-finetuned_full' 

See https://learn.microsoft.com/azure/storage/common/storage-use-azcopy-v10 for more information.
Uploading phi-4-mini-instruct-finetuned_full (4136.71 MBs): 100%|██████████| 4136708862/4136708862 [00:06<00:00, 643810623.03it/s]




Registered model: phi-4-mini-instruct-finetuned
Version: 1
ID: /subscriptions/6753a2ee-12b7-4fac-82fa-48824fb58abe/resourceGroups/rg-aml-yw-uno/providers/Microsoft.MachineLearningServices/workspaces/aml-workspace-yw-uno/models/phi-4-mini-instruct-finetuned/versions/1

Description:
Fine-tuned microsoft/Phi-4-mini-instruct with QLoRA + SFTTrainer
Dataset: ruslanmv/ai-medical-chatbot (1000 samples)
LoRA: r=8, alpha=32, dropout=0.05
Training: epochs=1, lr=0.0002, batch=1x8 grad_accum
Final train loss: 2.8415
Final eval loss: 2.7409

Tags: {'base_model': 'microsoft/Phi-4-mini-instruct', 'dataset': 'ruslanmv/ai-medical-chatbot', 'lora_r': '8', 'lora_alpha': '32', 'num_epochs': '1', 'quantization': 'QLoRA-4bit-NF4'}


## 11. Summary

### Model & Training Configuration
| Parameter | Value |
|-----------|-------|
| **Base Model** | `microsoft/Phi-4-mini-instruct` (3.8B params) |
| **Dataset** | `ruslanmv/ai-medical-chatbot` (1,000 samples: 900 train / 100 eval) |
| **Quantization** | QLoRA 4-bit NF4, double quant, fp16 compute dtype |
| **LoRA** | r=8, alpha=32, dropout=0.05, targets: `qkv_proj`, `o_proj`, `gate_up_proj`, `down_proj` |
| **Trainable Params** | 11.5M / 3.85B (0.30%) |
| **Training** | 1 epoch, lr=2e-4, batch=1×8 grad accum, paged AdamW 32-bit |
| **GPU** | NVIDIA Tesla T4 (15.6 GB VRAM, compute capability 7.5) |

### Training Results
| Metric | Step 50 | Step 100 | Eval |
|--------|---------|----------|------|
| **Loss** | 3.117 | 2.841 | 2.743 |
| **Token Accuracy** | 38.8% | 41.2% | 42.4% |
| **Entropy** | 2.941 | 2.856 | 2.814 |

- Loss consistently **decreased** (3.117 → 2.841 → 2.743 eval) — model is learning
- Eval loss < train loss — **no overfitting**
- Gradient norms stable (0.14 → 0.11) — numerically healthy training

### Saved Artifacts
| Artifact | Location |
|----------|----------|
| LoRA adapter + tokenizer | `/home/azureuser/localfiles/finetunedmodels/phi-4-mini-instruct-finetuned/` |
| LoRA config | `/home/azureuser/localfiles/finetunedmodels/phi-4-mini-instruct-finetuned_config/` |
| Merged full model + tokenizer | `/home/azureuser/localfiles/finetunedmodels/phi-4-mini-instruct-finetuned_full/` |
| AML Registered Model | `phi-4-mini-instruct-finetuned` (Azure ML workspace) |

### Key Takeaways
- **QLoRA + SFTTrainer** successfully fine-tuned Phi-4-mini-instruct on a T4 GPU within VRAM limits
- The model produces **coherent medical-style responses** aligned with the training dataset
- For further improvement: increase `NUM_DATA_ROWS` (5,000+) and `NUM_EPOCHS` (3+)
- MLflow tracking logged all metrics to the AML workspace for experiment comparison